In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Chargement des données
df = pd.read_csv('../data/finance_pme.csv')

print(f"Dataset chargé : {df.shape}")
print(f"\nPériode : {df['date'].iloc[0]} à {df['date'].iloc[-1]}")
print(f"CA total : {df['CA_total'].sum():,.0f} €")
print(f"Marge moyenne : {df['marge_brute_pct'].mean():.1f}%")
print(f"Trésorerie finale : {df['tresorerie'].iloc[-1]:,.0f} €")

# Visualisation 1 — Evolution CA et Charges
fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(
    go.Bar(x=df['date'], y=df['CA_total'],
           name='CA Total', marker_color='#3498DB'),
    secondary_y=False
)

fig1.add_trace(
    go.Bar(x=df['date'], y=df['charges_total'],
           name='Charges Total', marker_color='#E74C3C'),
    secondary_y=False
)

fig1.add_trace(
    go.Scatter(x=df['date'], y=df['marge_brute_pct'],
               name='Marge %', line=dict(color='#2ECC71', width=2),
               mode='lines+markers'),
    secondary_y=True
)

fig1.update_layout(
    title="Evolution CA, Charges et Marge — CôteAzur Boutique Hotel",
    barmode='group',
    xaxis_title="Mois",
    legend_title="Indicateurs"
)
fig1.update_yaxes(title_text="Montant (€)", secondary_y=False)
fig1.update_yaxes(title_text="Marge brute (%)", secondary_y=True)
fig1.show()

Dataset chargé : (24, 16)

Période : 2023-01 à 2024-12
CA total : 4,011,875 €
Marge moyenne : 17.3%
Trésorerie finale : 722,858 €


In [3]:
# Visualisation 2 — Trésorerie cumulée
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df['date'],
    y=df['tresorerie'],
    mode='lines+markers',
    name='Trésorerie cumulée',
    line=dict(color='#2ECC71', width=3),
    fill='tozeroy',
    fillcolor='rgba(46, 204, 113, 0.2)'
))

fig2.add_hline(
    y=0, 
    line_dash="dash", 
    line_color="red",
    annotation_text="Seuil critique"
)

fig2.update_layout(
    title="Evolution de la trésorerie cumulée — 2023/2024",
    xaxis_title="Mois",
    yaxis_title="Trésorerie (€)",
)
fig2.show()

# Visualisation 3 — Répartition des charges
charges_cols = ['charges_personnel', 'charges_fournisseurs', 
                'charges_energie', 'charges_marketing',
                'charges_loyer', 'charges_autres']

charges_total = df[charges_cols].sum()
charges_labels = ['Personnel', 'Fournisseurs', 'Energie', 
                  'Marketing', 'Loyer', 'Autres']

fig3 = px.pie(
    values=charges_total.values,
    names=charges_labels,
    title='Répartition des charges sur 2 ans',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_traces(textposition='inside', textinfo='percent+label')
fig3.show()

In [4]:
# Visualisation 4 — Forecast CA 2025
from scipy import stats

# Tendance linéaire sur 2023-2024
x = np.arange(len(df))
slope, intercept, r_value, p_value, std_err = stats.linregress(x, df['CA_total'])

# Projection 12 mois 2025
x_forecast = np.arange(len(df), len(df) + 12)
ca_forecast = slope * x_forecast + intercept

# Saisonnalité 2025
saisonnalite_2025 = [0.63, 0.68, 0.78, 0.88, 0.98, 1.23,
                     1.43, 1.38, 1.13, 0.88, 0.73, 0.78]
ca_forecast_saison = [ca * s for ca, s in zip(ca_forecast, saisonnalite_2025)]

dates_2025 = [f"2025-{str(i+1).zfill(2)}" for i in range(12)]

fig4 = go.Figure()

# Données historiques
fig4.add_trace(go.Scatter(
    x=df['date'],
    y=df['CA_total'],
    mode='lines+markers',
    name='CA réel 2023-2024',
    line=dict(color='#3498DB', width=2)
))

# Forecast 2025
fig4.add_trace(go.Scatter(
    x=dates_2025,
    y=ca_forecast_saison,
    mode='lines+markers',
    name='Forecast 2025',
    line=dict(color='#E74C3C', width=2, dash='dash'),
    marker=dict(symbol='diamond')
))

fig4.update_layout(
    title=f"CA réel vs Forecast 2025 — CôteAzur Boutique Hotel",
    xaxis_title="Mois",
    yaxis_title="CA (€)",
    legend_title="Période"
)
fig4.show()

print(f"\nCA forecast 2025 : {sum(ca_forecast_saison):,.0f} €")
print(f"Croissance estimée vs 2024 : +{((sum(ca_forecast_saison)/df[df['date'].str.startswith('2024')]['CA_total'].sum())-1)*100:.1f}%")


CA forecast 2025 : 2,104,864 €
Croissance estimée vs 2024 : +4.9%


## Synthèse financière — CôteAzur Boutique Hotel

### 📊 Indicateurs clés 2023-2024
- **CA total 2 ans : 4 011 875 €**
- **Marge brute moyenne : 17.3%**
- **Trésorerie finale : 722 858 €**
- **Croissance CA : +4.9% en 2025**

### 💡 Points d'attention
- Personnel = 42.6% des charges — optimisation possible hors saison
- Marge hivernale faible (12-14%) — diversifier les revenus
- Energie = 9.72% — potentiel d'économies avec investissements verts

### 🎯 Recommandations
- Renforcer les revenus restauration en basse saison
- Négocier les contrats fournisseurs annuellement
- Investir en marketing digital en janvier/février